# Missing Data Detective Work

Mastering missing data detection, analysis, and strategic handling

In [ ]:
import pandas as pd
import numpy as np

print("Missing data detective tools loaded!")

## Create Messy Dataset

In [ ]:
# Realistic patient data with missing values
patient_data = {
    'patient_id': ['P001', 'P002', 'P003', 'P004', 'P005', 'P006', 'P007', 'P008'],
    'age': [45, np.nan, 62, 34, np.nan, 58, 41, np.nan],
    'blood_pressure': [120, 135, np.nan, 118, 142, np.nan, 125, 130],
    'cholesterol': [200, np.nan, 185, np.nan, 220, 195, np.nan, 210],
    'test_date': ['2024-01-15', '2024-01-16', np.nan, '2024-01-18', '2024-01-19', np.nan, '2024-01-21', '2024-01-22']
}

df = pd.DataFrame(patient_data)
print("Raw patient data:")
print(df)

## Detect Missing Patterns

In [ ]:
# Count missing values per column
print("\nMissing values per column:")
print(df.isnull().sum())

# Calculate percentage missing
print("\nPercentage missing:")
print((df.isnull().sum() / len(df) * 100).round(1))

# Find rows with ANY missing values
print(f"\nRows with missing data: {df.isnull().any(axis=1).sum()} out of {len(df)}")

## Optional Visual Preview: Missingness Heatmap

The tabular summary above is the core Lecture 05 activity. This optional
preview turns the same missingness indicators into a heatmap for visual
learners. It previews the visualization workflow formalized in Lecture 07;
darker cells simply mean that a value is missing.

In [ ]:
missing_by_row = df.isna().sum(axis=1).rename('missing_fields')
print(pd.concat([df['patient_id'], missing_by_row], axis=1).to_string(index=False))

import matplotlib.pyplot as plt
import seaborn as sns

missing_indicator = df.drop(columns=['patient_id']).isna().astype(int)
figure, axis = plt.subplots(figsize=(8, 3))
sns.heatmap(
    missing_indicator.T,
    cmap=['#f2f2f2', '#d95f02'],
    cbar=False,
    linewidths=0.5,
    linecolor='white',
    ax=axis,
)
axis.set_title('Optional preview: where patient values are missing')
axis.set_xlabel('Patient row')
axis.set_ylabel('Field')
figure.tight_layout()
plt.show()

## Strategic Missing Data Handling

In [ ]:
# Strategy 1: Fill age with median (robust to outliers)
df['age_filled'] = df['age'].fillna(df['age'].median())
print("\nAge - filled with median:")
print(df[['patient_id', 'age', 'age_filled']])

# Strategy 2: parse and inspect dates; do not fill without a temporal contract.
parsed_dates = pd.to_datetime(df['test_date'], errors='coerce')
date_audit = pd.DataFrame({
    'patient_id': df['patient_id'],
    'raw_date': df['test_date'],
    'parsed_date': parsed_dates,
    'parse_failed': parsed_dates.isna() & df['test_date'].notna(),
})
print("\nParsed test dates and audit flags:")
print(date_audit)
print("Decision: retain missing dates; this fixture documents neither chronological row order nor an entity boundary for filling.")

# Strategy 3: Drop rows with critical missing data
# If BOTH blood_pressure AND cholesterol missing, row is useless
df_complete = df.dropna(subset=['blood_pressure', 'cholesterol'], how='all')
print(f"\nAfter dropping rows missing both BP and cholesterol: {len(df_complete)} rows remain")

## Compare Strategies

In [ ]:
print("\n=== SUMMARY OF STRATEGIES ===")
print(f"Original rows: {len(df)}")
print(f"Age: filled {df['age'].isnull().sum()} missing values with median")
print(f"Test dates: retained {parsed_dates.isna().sum()} missing dates pending temporal context")
print(f"Dropped {len(df) - len(df_complete)} rows with both BP and cholesterol missing")